### "A/B Testing & Experiment Analysis (Python)" in this python project "A" and "B" are :--
'Group A' was the control group that saw the existing checkout experience, while 'Group B' was the treatment group exposed to a new checkout design.

### ***-: EXPERIMENT DESIGN :-***

### *🔹1: Define the Experiment :-*

### Experiment Type
* Randomized controlled experiment
* Users are randomly assigned to Control (A) or Treatment (B)

### Duration
* 30 days (simulated)
*
### Unit of Analysis
* User-level
* Each user appears only once

---

### *🔹2: Define Primary & Secondary Metrics :-*

### Primary Metric (Decision Metric)
* Conversion Rate = Conversions / Total Users

### Secondary Metrics (Supporting Metrics)
* Revenue per User (RPU)
* Session Duration
* Conversion by device / country

---

### *🔹3: Hypothesis Definition :-*

### Null Hypothesis (H₀)
* The new checkout page does not change the conversion rate compared to the old checkout page.

[H₀: p_control = p_treatment]

### Alternative Hypothesis (H₁)
* The new checkout page increases the conversion rate compared to the old checkout page.

[H₁: p_treatment > p_control] - one side because business goal is improvement, not just difference.

---

### *🔹4: Significance Level (α) :-*
α = 0.05
### Interpretation
* 5% risk of false positive
* Industry standard
* Balanced risk for business decisions

---

### *🔹5: Decision Rule :-*
Perform a two-sample proportion z-test and
Compute a p-value

### Decision Criteria
* If p-value < 0.05 → Reject H₀ → Ship the change
* If p-value ≥ 0.05 → Fail to reject H₀ → Do not ship

---

### *🔹6: Assumptions Check :-*
### In this project -
* Random assignment
* Independent users
* Binary outcome
* Large enough sample size

In this simulated dataset satisfies all conditions.

### ***-: DOCUMENT HYPOTHESES :-***

### *Experiment Hypotheses :-*

*Null Hypothesis (H₀):*  
The conversion rate of the treatment group is equal to the conversion rate of the control group.

*Alternative Hypothesis (H₁):*  
The conversion rate of the treatment group is greater than the conversion rate of the control group.

**Significance Level:** α = 0.05

### ***-: PROJECT START :-***

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
# Create Raw Experiment Dataset (Uncleaned) :-

np.random.seed(42)
pd.set_option("display.float_format", "{:,.3f}".format)

n_users = 12000

user_ids = np.arange(100000, 100000 + n_users)

group = np.random.choice(
    ["Control", "Treatment"],
    size=n_users,
    p=[0.52, 0.48]  # slightly imbalanced (realistic)
)

# Conversion probability
conversion_prob = np.where(
    group == "Control", 0.105, 0.118
)

converted = np.random.binomial(1, conversion_prob)

# Revenue per user (only if converted)
revenue = np.where(
    converted == 1,
    np.round(np.random.gamma(shape=2, scale=45, size=n_users), 2),
    0
)

# Add outliers
revenue[np.random.choice(n_users, 30, replace=False)] *= 4

device = np.random.choice(
    ["Desktop", "Mobile", "Tablet"],
    size=n_users,
    p=[0.55, 0.40, 0.05]
)

country = np.random.choice(
    ["India", "USA", "UK", "Germany", "France"],
    size=n_users
)

session_duration = np.abs(
    np.random.normal(loc=180, scale=60, size=n_users)
)

# Create DataFrame
df = pd.DataFrame({
    "user_id": user_ids,
    "group": group,
    "converted": converted,
    "revenue": revenue,
    "device": device,
    "country": country,
    "session_duration_sec": session_duration
})


ASLO DOWNLOADED AS "AB_test_data.csv"

In [ ]:
#Checking Dataset :-
df.head()

In [ ]:
#Initial Sanity Check :-
df.info()

### ***-: DATA VALIDATION & EXPERIMENT SANITY CHECKS :-***

In [ ]:
#1. Group Size Balance Check :-

group_counts = df["group"].value_counts()
group_counts

In [ ]:
#2. Randomization Check (Device Distribution) :-

pd.crosstab(df["group"], df["device"], normalize="index")

In [ ]:
#3. Randomization Check (Country Distribution) :-

pd.crosstab(df["group"], df["country"], normalize="index")

In [ ]:
#4. Session Duration Balance :-

df.groupby("group")["session_duration_sec"].describe()

# Visual
plt.figure(figsize=(8,5))
sns.boxplot(
    x="group",
    y="session_duration_sec",
    data=df
)
plt.title("Session Duration Distribution by Group")
plt.show()

In [ ]:
#5. Conversion Rate Quick Look :-

conversion_rates = df.groupby("group")["converted"].mean()
conversion_rates

In [ ]:
#6. Revenue Distribution Check :-

df.groupby("group")["revenue"].describe()

In [ ]:
#7. Data Integrity Checks (Duplicate) :-

df["user_id"].duplicated().sum()

In [ ]:
#8. Data Integrity Checks (Missing value) :-

df.isna().sum()

### ***-: SANITY CHECK SUMMARY :-***

Experiment Sanity Checks

- Control and Treatment groups have comparable sizes.
- Device and country distributions are balanced across groups.
- Session duration distributions are similar, indicating comparable user behavior.
- No missing or duplicate user records were found.
- The experiment appears properly randomized and suitable for hypothesis testing.

### ***-: EXPLORATORY ANALYSIS (Conversion & Revenue) :-***

In [ ]:
#1. Conversion Rate Comparison :-
conversion_summary = (
    df.groupby("group")["converted"]
    .mean()
    .reset_index()
)

conversion_summary

In [ ]:
# Visualize :-
plt.figure(figsize=(6,4))
sns.barplot(
    data=conversion_summary,
    x="group",
    y="converted"
)
plt.title("Conversion Rate by Group")
plt.ylabel("Conversion Rate")
plt.show()

In [ ]:
#2. Conversion by Device :-
device_conversion = (
    df.groupby(["group", "device"])["converted"]
    .mean()
    .reset_index()
)

device_conversion

In [ ]:
# Visualize :-
plt.figure(figsize=(8,5))
sns.barplot(
    data=device_conversion,
    x="device",
    y="converted",
    hue="group"
)
plt.title("Conversion Rate by Device and Group")
plt.show()

In [ ]:
#3. Revenue per User (RPU) :-

rpu = (
    df.groupby("group")["revenue"]
    .mean()
    .reset_index()
)

rpu

In [ ]:
#4. Revenue Distribution :-

plt.figure(figsize=(8,5))
sns.boxplot(
    data=df[df["converted"] == 1],
    x="group",
    y="revenue"
)
plt.title("Revenue Distribution (Converted Users)")
plt.show()

### ***-: EXPLORATORY INSIGHTS :-***

Observations

- The treatment group shows a slightly higher conversion rate compared to the control group.
- Conversion behavior varies across devices, suggesting potential segment-level effects.
- Revenue per user appears marginally higher for the treatment group, though revenue distribution is highly skewed.
- Converted users tend to have longer session durations, indicating higher engagement.
- Visual differences suggest potential improvement, but statistical testing is required for confirmation.


### ***-: STATISTICAL TESTING (Conversion Rate) :-***

In [ ]:
#1. Prepare Conversion Data :-

conversion_counts = df.groupby("group")["converted"].agg(
    conversions="sum",
    users="count"
)

conversion_counts

In [ ]:
#2. Extract Values for Testing :-

conversions = conversion_counts["conversions"].values
users = conversion_counts["users"].values

conversions, users

In [ ]:
#3. Two-Sample Proportion Z-Test :-

z_stat, p_value = proportions_ztest(
    count=conversions,
    nobs=users,
    alternative="smaller"  # Control < Treatment
)

z_stat, p_value

In [ ]:
#4. Decision Rule :-

alpha = 0.05

if p_value < alpha:
    decision = "Reject H₀ — Treatment improves conversion"
else:
    decision = "Fail to reject H₀ — No significant improvement"

decision

### ***-: STATISTICAL CONCLUSION :-***

Conversion Rate Hypothesis Test

- A two-sample proportion z-test was conducted to compare conversion rates between the control and treatment groups.

- The resulting p-value was evaluated against a significance level of 0.05. Based on this test, we conclude that the difference in conversion rates is statistically significant, indicating that the treatment checkout page leads to improved conversion performance.


### ***-: REVENUE IMPACT ANALYSIS & STATISTICAL TESTING :-***

In [ ]:
#1. Revenue per User (RPU) :-

rpu

In [ ]:
#2. Visual Comparison :-

plt.figure(figsize=(8,5))
sns.boxplot(
    data=df,
    x="group",
    y="revenue"
)
plt.title("Revenue per User Distribution by Group")
plt.show()

In [ ]:
#3. Two-Sample t-test :-

from scipy.stats import ttest_ind

control_rev = df[df["group"] == "Control"]["revenue"]
treatment_rev = df[df["group"] == "Treatment"]["revenue"]

t_stat, p_value_ttest = ttest_ind(
    treatment_rev,
    control_rev,
    equal_var=False
)

t_stat, p_value_ttest

In [ ]:
#4. Mann–Whitney U Test :-

from scipy.stats import mannwhitneyu

u_stat, p_value_mwu = mannwhitneyu(
    treatment_rev,
    control_rev,
    alternative="greater"
)

u_stat, p_value_mwu

In [ ]:
#5. Decision Summary :-

alpha = 0.05

if p_value_mwu < alpha:
    revenue_decision = "Treatment generates significantly higher revenue"
else:
    revenue_decision = "No statistically significant revenue uplift"

revenue_decision

### ***-: REVENUE CONCLUSION :-***

Revenue Impact Analysis

- Revenue per user was compared between control and treatment groups using both a Welch’s t-test and a Mann–Whitney U test to account for skewed and zero-inflated revenue data.

- Results indicate that the treatment group generates higher average revenue per user, and this difference is statistically significant. This suggests that the new checkout experience not only improves conversion but also drives incremental revenue.

### ***-: FINAL BUSINESS RECOMMENDATION :-***

**Final Recommendation**

Based on the A/B experiment results, the treatment (new checkout experience) demonstrated a statistically significant improvement in conversion rate compared to the control group.

Additionally, revenue per user analysis showed that the treatment group generated higher average revenue, and this uplift was validated using both parametric and non-parametric statistical tests.

**Recommendation:**
- Roll out the new checkout experience to 100% of users.
- Monitor post-launch metrics to ensure sustained performance.
- Use this experiment as a baseline framework for future UX experiments.

---

### ***-: RISK ASSESSMENT & LIMITATIONS :-***

**Risk Assessment & Limitations**

- Revenue data is highly skewed and contains extreme outliers, which may
  influence mean-based metrics.
- The experiment duration may not capture long-term user behavior changes.
- Results assume proper randomization and no external traffic bias.
- Segment-level impacts (device, geography) should be further analyzed
  before localized rollouts.

Despite these limitations, the consistency of results across multiple
statistical tests increases confidence in the observed uplift.

---